# Old vs new cellina metrics

Old numbers: `results/loo_summary_{dataset_name}_DEG_50_v3.csv`. New numbers: JSONs in `DATA_ROOT/datasets/{dataset_name}/correlations` (read as in `ood_plots.ipynb`).

- edge: `cellina` (csv) vs `cellina-cr-cf` (json)
- node: `cellina-pert` (csv) vs `cellina-cr-cf-node` (json)

In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)

In [2]:
dataset_name = "merfish"
DATA_ROOT = '/data/a330d'  # JSON_BASE_PATH
SUMMARY_CSV = f'../loo_summary_{dataset_name}_DEG_50_v4.csv'

# old model name (csv) -> new model name (json)
PAIRS = {
    'cellina': 'cellina-cr-cf',
    'cellina-pert': 'cellina-cr-cf-node',
    'cellina-graph': 'cellina-graph-cr-cf',
    'cellina-gat-pert': 'cellina-graph-cr-cf-node',
}
METRICS = ['pearson', 'direction_match_k', 'edistance_pca_log', 'rmse']
KEYS = ['sid', 'holdout_celltype']

## Load

In [3]:
df_old = pd.read_csv(SUMMARY_CSV)
df_old = df_old[df_old['model_name'].isin(PAIRS.keys())]
df_old.groupby('model_name').size()

model_name
cellina             30
cellina-gat-pert    30
cellina-graph       30
cellina-pert        30
dtype: int64

In [4]:
corr_dir = os.path.join(DATA_ROOT, f"datasets/{dataset_name}/correlations")
pattern = os.path.join(corr_dir, "*.json")
files = sorted(glob.glob(pattern))

rows = []
for fp in files:
    name = os.path.basename(fp)
    core = name[len("crc_"):-len(".json")] if dataset_name == "crc" else name[:-len(".json")]
    parts = core.split("_")
    sid = parts[0]
    model_name = parts[1]
    holdout_celltype = "_".join(parts[2:])
    if model_name not in PAIRS.values():
        continue
    with open(fp, "r") as f:
        data = json.load(f)
    rows.append({
        "sid": f"crc_{sid}" if dataset_name == "crc" else sid,
        "model_name": model_name,
        "holdout_celltype": holdout_celltype,
        **{m: data.get(m) for m in METRICS},
    })

df_new = pd.DataFrame(rows)
df_new.groupby('model_name').size()

model_name
cellina-cr-cf               30
cellina-cr-cf-node          30
cellina-graph-cr-cf         30
cellina-graph-cr-cf-node    30
dtype: int64

## Side-by-side per run

Rows: runs present in both csv and json. Columns: metric → old | new | Δ (new − old).

In [5]:
def side_by_side(old_name, new_name):
    old = df_old[df_old['model_name'] == old_name].set_index(KEYS)[METRICS]
    new = df_new[df_new['model_name'] == new_name].set_index(KEYS)[METRICS]
    old, new = old.align(new, join='inner')
    out = pd.concat({m: pd.DataFrame({f'old ({old_name})': old[m],
                                      f'new ({new_name})': new[m],
                                      'Δ': np.round(new[m] - old[m], 2)}) for m in METRICS}, axis=1)
    return out

comparisons = {old_name: side_by_side(old_name, new_name) for old_name, new_name in PAIRS.items()}

In [6]:
comparisons['cellina'].round(3)

pearson                           direction_match_k                           edistance_pca_log                                    rmse  \
                                                old (cellina) new (cellina-cr-cf)     Δ     old (cellina) new (cellina-cr-cf)     Δ     old (cellina) new (cellina-cr-cf)     Δ old (cellina)   
sid           holdout_celltype                                                                                                                                                                  
C57BL6J-2.036 GABAergic neuron_Fiber_tracts             0.914               0.917  0.00              0.64                0.66  0.02             9.546              10.127  0.58       840.449   
              GABAergic neuron_Isocortex                0.971               0.974  0.00              0.68                0.70  0.02            10.250              10.107 -0.14      1317.795   
              astrocyte_Fiber_tracts                    0.487               0.480 -0.01              0.24                0.26  0.02             8.039               7.408 -0.63      3937.514   
              astrocyte_Isocortex                       0.868               0.873  0.01              0.36                0.40  0.04             8.845               8.929  0.08      3284.066   
              endothelial cell_Fiber_tracts             0.994               0.992 -0.00              0.14                0.14  0.00             6.489               6.649  0.16      3919.867   
              endothelial cell_Isocortex                0.932               0.934  0.00              0.72                0.70 -0.02             7.140               7.670  0.53      2769.361   
              glutamatergic neuron_Fiber_tracts         0.892               0.917  0.03              0.48                0.50  0.02            10.710              10.346 -0.36     12015.626   
              glutamatergic neuron_Isocortex            0.951               0.974  0.02              0.46                0.68  0.22            10.100               7.513 -2.59     14819.967   
              oligodendrocyte_Fiber_tracts              0.659               0.652 -0.01              0.42                0.42  0.00             6.756               6.651 -0.11      6741.057   
              oligodendrocyte_Isocortex                 0.518               0.516 -0.00              0.48                0.46 -0.02             6.425               6.954  0.53      6634.517   
C57BL6J-2.039 GABAergic neuron_Fiber_tracts             0.798               0.809  0.01              0.32                0.34  0.02             9.022               9.371  0.35      1662.413   
              GABAergic neuron_Isocortex                0.961               0.964  0.00              0.46                0.48  0.02             8.280               9.114  0.83      1499.806   
              astrocyte_Fiber_tracts                    0.783               0.796  0.01              0.58                0.66  0.08             6.553               6.393 -0.16      4729.214   
              astrocyte_Isocortex                       0.888               0.896  0.01              0.62                0.66  0.04             6.842               7.137  0.29      3733.379   
              endothelial cell_Fiber_tracts             0.994               0.993 -0.00              0.12                0.12  0.00             7.258               7.454  0.20      5117.989   
              endothelial cell_Isocortex                0.876               0.881  0.01              0.66                0.62 -0.04             6.393               6.995  0.60      3492.740   
              glutamatergic neuron_Fiber_tracts         0.807               0.884  0.08              0.46                0.46  0.00            10.437              10.336 -0.10     15796.039   
              glutamatergic neuron_Isocortex            0.967               0.985  0.02              0.56                0.70  0.14             7.845               6.338 -1.51     13638.788   
 

In [7]:
comparisons['cellina-pert'].round(3)

pearson                                 direction_match_k                                 edistance_pca_log                                 \
                                                old (cellina-pert) new (cellina-cr-cf-node)     Δ old (cellina-pert) new (cellina-cr-cf-node)     Δ old (cellina-pert) new (cellina-cr-cf-node)     Δ   
sid           holdout_celltype                                                                                                                                                                          
C57BL6J-2.036 glutamatergic neuron_Isocortex                 0.962                    0.949 -0.01               0.52                     0.44 -0.08             11.616                   15.325  3.71   
              glutamatergic neuron_Fiber_tracts              0.860                    0.852 -0.01               0.42                     0.34 -0.08             14.554                   17.861  3.31   
              oligodendrocyte_Isocortex                      0.502                    0.495 -0.01               0.38                     0.40  0.02              7.805                    6.649 -1.16   
              oligodendrocyte_Fiber_tracts                   0.666                    0.697  0.03               0.28                     0.56  0.28              8.859                    6.107 -2.75   
              astrocyte_Isocortex                            0.896                    0.835 -0.06               0.52                     0.30 -0.22              7.878                    8.799  0.92   
              astrocyte_Fiber_tracts                         0.501                    0.495 -0.01               0.32                     0.18 -0.14              8.205                    8.254  0.05   
              GABAergic neuron_Isocortex                     0.954                    0.940 -0.01               0.56                     0.58  0.02             12.888                   12.768 -0.12   
              GABAergic neuron_Fiber_tracts                  0.926                    0.916 -0.01               0.76                     0.68 -0.08              9.299                    9.264 -0.03   
              endothelial cell_Isocortex                     0.927                    0.931  0.00               0.62                     0.68  0.06              9.162                    6.837 -2.33   
              endothelial cell_Fiber_tracts                  0.994                    0.995  0.00               0.14                     0.16  0.02              8.505                    6.663 -1.84   
C57BL6J-2.039 glutamatergic neuron_Isocortex                 0.970                    0.973  0.00               0.60                     0.46 -0.14             10.265                   13.131  2.87   
              glutamatergic neuron_Fiber_tracts              0.812                    0.697 -0.11               0.42                     0.28 -0.14             12.436                   13.831  1.40   
              oligodendrocyte_Isocortex                      0.595                    0.591 -0.00               0.56                     0.54 -0.02              7.298                    6.595 -0.70   
              oligodendrocyte_Fiber_tracts                   0.752                    0.778  0.03               0.60                     0.48 -0.12              8.145                    6.868 -1.28   
              astrocyte_Isocortex                            0.881                    0.856 -0.02               0.56                     0.50 -0.06              7.861                    7.678 -0.18   
              astrocyte_Fiber_tracts                         0.794                    0.775 -0.02               0.60                     0.48 -0.12              7.393                    6.718 -0.67   
              GABAergic neuron_Isocortex                     0.926                    0.893 -0.03               0.44                     0.34 -0.10              8.564                    9.891  1.33   
              GABAergic neuron_Fiber_tr

## Mean over matched runs

Rows: metrics. Columns: old | new | Δ per pair.

In [8]:
summary = pd.concat({f'{o} → {n}': comparisons[o].mean().unstack().reindex(columns=comparisons[o].columns.get_level_values(1).unique())
                     .set_axis(['old', 'new', 'Δ'], axis=1)
                     for o, n in PAIRS.items()}, axis=1)
summary.round(2)

cellina → cellina-cr-cf                  cellina-pert → cellina-cr-cf-node                 cellina-graph → cellina-graph-cr-cf                   \
                                      old      new       Δ                               old      new      Δ                                 old      new       Δ   
pearson                              0.83     0.84    0.01                              0.82     0.82  -0.01                                0.83     0.84    0.01   
direction_match_k                    0.47     0.50    0.03                              0.47     0.42  -0.04                                0.49     0.53    0.04   
edistance_pca_log                    7.96     7.94   -0.02                              9.08     8.98  -0.09                                8.07     7.67   -0.41   
rmse                              5721.14  5431.60 -289.55                           5622.08  5719.07  97.00                             5498.67  5262.40 -236.27   

                  cellina-gat-pert → cellina-graph-cr-cf-node                   
                                                          old      new       Δ  
pearson                                                  0.83     0.83    0.01  
direction_match_k                                        0.47     0.51    0.04  
edistance_pca_log                                        8.11     8.61    0.51  
rmse                                                  5669.65  5334.10 -335.54